### PPO（Proximal Policy Optimization，近端策略优化）
与Q-Learning和DQN是“基于价值（Value-Based）”不同的是，PPO是“基于策略（Policy-Based）”的方法
其构建在Actor-Critic（演员-评论家）的架构之上


**Actor-Critic架构**
- **演员网络 Actor**：负责决策，它根据当前状态（State）输出一个动作（Action）或动作的概率分布
- **Critic（评论家网络）**：负责评估。它根据当前状态估计一个价值（Value），即从这个状态出发，预期能获得多少累计奖励

**优势函数(Advantage Function)**
PPO并不直接使用原始奖励来指导学习，而是使用优势函数（Advantage Function），其衡量的是：在某个状态下，采取某个特定动作，比这个状态的平均水平好多少。实践中，PPO常使用GAE（Generalized Advantage Estimation，广义优势估计） 来计算优势函数，它能在偏差和方差之间取得更好的平衡
- 优势值为正，意味着这个动作优于平均水平，应该鼓励它
- 优势值为负，意味着这个动作差于平均水平，应该抑制它

**核心思想**
PPO通过一个裁剪（Clipping）机制，将策略更新的幅度限制在一个“安全”的范围内。
min操作，使得PPO最终选择裁剪后的目标和未裁剪的目标中较小的那个，不会过度依赖那些可能导致大幅策略偏离的更新
$$
L^{CLIP}(\theta) = \hat{\mathbb{E}}_t \left[ \min \left( r_t(\theta) \hat{A}_t, \operatorname{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]
$$

- **$\hat{\mathbb{E}}_t$**：表示对一批（Mini-batch）经验样本的经验平均期望。
- **$r_t(\theta)$**：**概率比（Probability Ratio）**，定义为 $\frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{\text{old}}}(a_t | s_t)}$。它衡量了新策略与旧策略在动作 $a_t$ 上选择概率的差异。
- **$\hat{A}_t$**：**优势函数（Advantage Function）** 的估计值，衡量在状态 $s_t$ 下选择动作 $a_t$ 相比平均水平好多少（正数代表好，负数代表差）。
- **$\epsilon$**：**剪切超参数（Clip Hyperparameter）**，通常取 $0.1$ 或 $0.2$，定义了策略更新的“安全区间” $[1-\epsilon, 1+\epsilon]$。
- **$\operatorname{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)$**：将概率比 $r_t(\theta)$ 强制限制在区间 $[1-\epsilon, 1+\epsilon]$ 内的操作。

**缺点**
- **仍为同策略（On-Policy）算法**：用于更新的数据必须由当前最新的策略生成，无法直接复用旧策略的历史数据
- **计算资源要求**：在大规模、复杂的环境中训练，仍需较高的计算资源


**应用场景**
- **游戏AI**：如OpenAI在《Dota2》和《星际争霸》中的应用
- **机器人控制**：如机器人步态学习
- **大语言模型（LLM）对齐**：它是在RLHF（基于人类反馈的强化学习）框架中，用于微调语言模型的核心算法之一

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import matplotlib
matplotlib.use('Agg')  # 避免绘图后端冲突

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.distributions import Categorical

# ==================== 1. 设置设备与超参数 ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 环境
env = gym.make('CartPole-v1')
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n

# PPO 超参数
EPISODES = 2000                # 总训练回合数
STEPS_PER_UPDATE = 2048        # 每次更新收集的步数（相当于rollout长度）
GAMMA = 0.99                   # 折扣因子
GAE_LAMBDA = 0.95              # GAE 平滑参数
CLIP_RANGE = 0.2               # 裁剪范围 ε
LR_ACTOR = 3e-4                # Actor 学习率
LR_CRITIC = 1e-3               # Critic 学习率
ENTROPY_COEF = 0.01            # 熵奖励系数
BATCH_SIZE = 64                # 小批量大小
PPO_EPOCHS = 10                # 每批数据训练的轮数

# ==================== 2. Actor-Critic 网络 ====================
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        # 共享特征提取层
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh()
        )
        # Actor 头（输出动作概率的logits）
        self.actor = nn.Linear(128, act_dim)
        # Critic 头（输出状态价值 V(s)）
        self.critic = nn.Linear(128, 1)

    def forward(self, obs):
        features = self.shared(obs)
        return self.actor(features), self.critic(features)

    def get_action_logits(self, obs):
        features = self.shared(obs)
        return self.actor(features)

    def get_value(self, obs):
        features = self.shared(obs)
        return self.critic(features)

# 初始化网络与优化器
policy = ActorCritic(obs_dim, act_dim).to(device)
optimizer = optim.Adam([
    {'params': policy.shared.parameters(), 'lr': LR_ACTOR},
    {'params': policy.actor.parameters(), 'lr': LR_ACTOR},
    {'params': policy.critic.parameters(), 'lr': LR_CRITIC}
])

# ==================== 3. 经验收集函数 ====================
def collect_trajectories(env, policy, steps):
    """与环境交互，收集一条轨迹（rollout）"""
    states, actions, rewards, dones, log_probs, values = [], [], [], [], [], []
    state, _ = env.reset()
    state = state.astype(np.float32)
    done = False

    for _ in range(steps):
        state_t = torch.tensor(state, device=device).unsqueeze(0)
        with torch.no_grad():
            logits = policy.get_action_logits(state_t)
            value = policy.get_value(state_t)
            dist = Categorical(logits=logits)
            action = dist.sample()
            log_prob = dist.log_prob(action)

        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        next_state = next_state.astype(np.float32) if not done else None

        # 存储经验
        states.append(state)
        actions.append(action.item())
        rewards.append(reward)
        dones.append(done)
        log_probs.append(log_prob.item())
        values.append(value.item())

        # 状态转移
        state = next_state if not done else env.reset()[0].astype(np.float32)

    return (np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(dones, dtype=np.float32),
            np.array(log_probs, dtype=np.float32),
            np.array(values, dtype=np.float32))

# ==================== 4. 计算 GAE（广义优势估计） ====================
def compute_gae(rewards, dones, values, gamma, gae_lambda):
    """计算优势函数 A(s,a) 和回报 G_t"""
    advantages = np.zeros_like(rewards, dtype=np.float32)
    last_gae = 0.0
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_value = 0.0  # 终止状态之后的价值为 0
        else:
            next_value = values[t + 1]
        # TD误差：δ = r + γ * V(s') - V(s)
        delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]
        # GAE 累计
        last_gae = delta + gamma * gae_lambda * (1 - dones[t]) * last_gae
        advantages[t] = last_gae
    # 回报 G_t = A_t + V(s_t)
    returns = advantages + values
    return advantages, returns

# ==================== 5. PPO 核心更新步骤 ====================
def ppo_update(policy, optimizer, states, actions, old_log_probs, advantages, returns, epochs, batch_size, clip_range, entropy_coef):
    """执行 PPO 的裁剪优化"""
    # 标准化优势（稳定训练）
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    dataset_size = len(states)
    indices = np.arange(dataset_size)

    for _ in range(epochs):
        np.random.shuffle(indices)
        for start in range(0, dataset_size, batch_size):
            end = start + batch_size
            batch_indices = indices[start:end]

            # 提取 batch 数据
            batch_states = torch.tensor(states[batch_indices], device=device)
            batch_actions = torch.tensor(actions[batch_indices], device=device)
            batch_old_log_probs = torch.tensor(old_log_probs[batch_indices], device=device)
            batch_advantages = torch.tensor(advantages[batch_indices], device=device)
            batch_returns = torch.tensor(returns[batch_indices], device=device)

            # 1. 使用当前策略计算新的 log_probs 和 value
            logits = policy.get_action_logits(batch_states)
            dist = Categorical(logits=logits)
            new_log_probs = dist.log_prob(batch_actions)
            values = policy.get_value(batch_states).squeeze(1)

            # 2. 计算概率比 r_t(θ)
            ratio = torch.exp(new_log_probs - batch_old_log_probs)

            # 3. PPO 裁剪损失（核心公式）
            surr1 = ratio * batch_advantages
            surr2 = torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * batch_advantages
            policy_loss = -torch.min(surr1, surr2).mean()

            # 4. 价值损失（MSE，用于更新 Critic）
            value_loss = nn.MSELoss()(values, batch_returns)

            # 5. 熵奖励（鼓励探索）
            entropy = dist.entropy().mean()
            entropy_loss = -entropy_coef * entropy

            # 总损失
            loss = policy_loss + value_loss + entropy_loss

            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            # 梯度裁剪（防止梯度爆炸）
            torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=0.5)
            optimizer.step()

# ==================== 6. 训练主循环 ====================
episode_rewards = []
total_steps = 0

for episode in range(EPISODES):
    # 收集一批经验
    states, actions, rewards, dones, old_log_probs, values = collect_trajectories(
        env, policy, STEPS_PER_UPDATE
    )

    # 计算优势和回报
    advantages, returns = compute_gae(rewards, dones, values, GAMMA, GAE_LAMBDA)

    # PPO 更新
    ppo_update(
        policy, optimizer,
        states, actions, old_log_probs, advantages, returns,
        epochs=PPO_EPOCHS,
        batch_size=BATCH_SIZE,
        clip_range=CLIP_RANGE,
        entropy_coef=ENTROPY_COEF
    )

    # 记录本回合平均奖励（用于监控）
    total_steps += STEPS_PER_UPDATE
    episode_reward = np.sum(rewards)  # 这是本轮 collected 步数的总奖励
    episode_rewards.append(episode_reward)

    if (episode+1) % 50 == 0:
        avg_reward = np.mean(episode_rewards[-50:])
        print(f"Episode {episode+1}/{EPISODES}, Avg Reward (last 50): {avg_reward:.1f}, Total Steps: {total_steps}")

    # 早期停止（如果连续平均超过190，可视为解决）
    if len(episode_rewards) >= 50 and np.mean(episode_rewards[-50:]) >= 190:
        print(f"Solved! Stopping early at episode {episode+1}")
        break

# ==================== 7. 测试与可视化 ====================
print("Training finished. Testing...")

# 测试保存的模型（渲染画面）
test_env = gym.make('CartPole-v1', render_mode='human')
state, _ = test_env.reset()
state = state.astype(np.float32)
total_test_reward = 0
for _ in range(500):
    state_t = torch.tensor(state, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = policy.get_action_logits(state_t)
        action = logits.argmax().item()
    next_state, reward, terminated, truncated, _ = test_env.step(action)
    state = next_state.astype(np.float32)
    total_test_reward += reward
    if terminated or truncated:
        state, _ = test_env.reset()
        state = state.astype(np.float32)
test_env.close()
print(f"Test total reward (single episode): {total_test_reward}")

# 保存训练曲线
plt.plot(episode_rewards)
plt.xlabel('Update Batch (each batch = 2048 steps)')
plt.ylabel('Total Reward in Batch')
plt.title('PPO on CartPole (Custom Implementation)')
plt.savefig('ppo_result.png', dpi=150)
print("训练曲线已保存为 ppo_result.png")

ModuleNotFoundError: No module named 'gymnasium'